# PEAK M-ATH — Abnormal DLL Loading in Common Applications

Hunt for unusual DLL or module loads initiated by high-prevalence applications such as **Microsoft Teams** and **Microsoft Edge** (DLL side-loading, search-order abuse, blending with trusted processes), following the **PEAK** framework: **Prepare → Execute → Act → Knowledge**.

**Ref:** M25 — Abnormal DLL loading in common applications  
**M-ATH approach:** Fleet-rarity scoring combined with path heuristics and optional VirusTotal hash reputation to surface uncommon module loads.

## How to use
1. Put module-load CSV exports into `input/` — **per-event** columns such as Sysmon `Image` / `ImageLoaded`, or **aggregated** SentinelOne Power Query exports such as `module.path`, `module.sha1`, `count()` (no process column; scope Teams/Edge in the query)
2. Optionally set `VT_API_KEY` in `.env` for file hash reputation (SHA-256, SHA-1, or MD5)
3. Run all cells
4. Review ranked findings in `output/`


In [9]:
pass  # Placeholder (removed environment-specific output)


## PREPARE — Plan your Approach

- **Select Topic:** Abnormal DLL loading in common applications — adversaries side-load malicious DLLs through trusted high-prevalence processes (MITRE ATT&CK [T1574.002](https://attack.mitre.org/techniques/T1574/002/) Hijack Execution Flow: DLL Side-Loading).
- **Research Topic:** DLL side-loading techniques, fleet-rarity analysis, path heuristics for suspicious module locations, VirusTotal file hash reputation.
- **Identify Datasets:** Module-load telemetry (Sysmon, SentinelOne Power Query) with process name, module path, hash, and optional event counts.
- **Select Algorithms:** Fleet-rarity counting (rare modules across endpoints), path-based heuristic scoring (temp dirs, user-writable locations, unsigned paths), optional VT hash lookup, exclusion overlay.

In [10]:
# Scenario mode: anchor paths to this notebook's scenario folder.
import os
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    while cur != cur.parent:
        if (cur / 'detection_logics').exists() and (cur / 'scenarios').exists():
            return cur
        cur = cur.parent
    raise RuntimeError('Unable to locate repository root from current working directory.')

REPO_ROOT = find_repo_root(Path.cwd())
SCENARIO_DIR = REPO_ROOT / 'scenarios' / 'abnormal_dll_loading_common_applications'
if not SCENARIO_DIR.exists():
    raise FileNotFoundError(f'Scenario folder not found: {SCENARIO_DIR.relative_to(REPO_ROOT)}')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

INPUT_DIR = SCENARIO_DIR / 'input'
OUTPUT_DIR = SCENARIO_DIR / 'output'
REPO_EXCLUSIONS_DIR = REPO_ROOT / 'exclusions'
SCENARIO_EXCLUSIONS_DIR = SCENARIO_DIR / 'exclusions'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCENARIO_MODE = True
print(f'Scenario folder: {SCENARIO_DIR.relative_to(REPO_ROOT)}')


Scenario folder: scenarios\abnormal_dll_loading_common_applications


In [11]:
import glob
import json
import math
import os
import re
import time
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Image

pd.set_option('display.max_colwidth', 180)

if not globals().get('SCENARIO_MODE', False):
    INPUT_DIR = Path('input')
    OUTPUT_DIR = Path('output')
    REPO_EXCLUSIONS_DIR = Path('exclusions')
    SCENARIO_EXCLUSIONS_DIR = Path('exclusions')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXCLUDED_VALUES_REPO = REPO_EXCLUSIONS_DIR / 'excluded_values.conf'
EXCLUDED_REASONS_REPO = REPO_EXCLUSIONS_DIR / 'excluded_values+reasons.conf'
EXCLUDED_VALUES_SCENARIO = SCENARIO_EXCLUSIONS_DIR / 'excluded_values.conf'
EXCLUDED_REASONS_SCENARIO = SCENARIO_EXCLUSIONS_DIR / 'excluded_values+reasons.conf'

LOW_PREVALENCE_THRESHOLD = 2
SCORE_THRESHOLD = 2

# Basenames (lowercase) for loading processes to include (edit for your environment)
TARGET_PROCESS_NAMES = frozenset({
    'teams.exe', 'ms-teams.exe', 'msedge.exe', 'chrome.exe', 'outlook.exe',
})

def _rel(p):
    try:
        if globals().get('SCENARIO_MODE', False) and 'REPO_ROOT' in globals():
            root = globals()['REPO_ROOT']
            if hasattr(p, 'is_relative_to') and p.is_relative_to(root):
                return p.relative_to(root)
        return p
    except (ValueError, AttributeError):
        return p

print(f'Input folder: {_rel(INPUT_DIR)}')
print(f'Output folder: {_rel(OUTPUT_DIR)}')
print(f'Exclusions (repo): {_rel(EXCLUDED_VALUES_REPO)}; scenario: {_rel(EXCLUDED_VALUES_SCENARIO)}')


Input folder: scenarios\abnormal_dll_loading_common_applications\input
Output folder: scenarios\abnormal_dll_loading_common_applications\output
Exclusions (repo): exclusions\excluded_values.conf; scenario: scenarios\abnormal_dll_loading_common_applications\exclusions\excluded_values.conf


## Path normalization and scoring

Heuristic scoring (no ML dependency): temp and user download paths, non-standard roots, high file-name entropy, **fleet-wide rarity** of the loaded module path, optional low counts from a vendor prevalence column, and optional **VirusTotal** file verdict when `VT_API_KEY` and SHA-256 are available.

Exclusions merge **repository** `exclusions/` and this scenario's `exclusions/` (same pattern as other scenarios).


In [12]:
def shannon_entropy(text: str) -> float:
    text = (text or '').strip()
    if not text:
        return 0.0
    probs = [text.count(ch) / len(text) for ch in set(text)]
    return -sum(p * math.log2(p) for p in probs if p > 0)


def normalize_windows_path(value: str) -> str:
    s = str(value or '').strip().strip('"')
    s = s.replace('/', '\\')
    return s.lower()


def path_basename(path: str) -> str:
    p = normalize_windows_path(path)
    if not p:
        return ''
    return p.split('\\')[-1]


def read_conf_lines(path: Path):
    if not path.exists():
        return []
    lines = []
    with path.open('r', encoding='utf-8') as handle:
        for raw in handle:
            line = raw.split('#', 1)[0].strip()
            if not line:
                continue
            lines.append(line)
    return lines


def normalize_exclusion_value(value):
    return str(value or '').strip().lower()


def load_exclusions_from_files(values_file: Path, value_reason_file: Path):
    excluded_values = {
        normalize_exclusion_value(v)
        for v in read_conf_lines(values_file)
        if normalize_exclusion_value(v)
    }
    excluded_pairs = set()
    for line in read_conf_lines(value_reason_file):
        if '||' not in line:
            continue
        value_part, reason_part = line.split('||', 1)
        v_norm = normalize_exclusion_value(value_part)
        r_norm = str(reason_part or '').strip().lower()
        if v_norm and r_norm:
            excluded_pairs.add((v_norm, r_norm))
    return excluded_values, excluded_pairs


def merge_exclusions():
    v1, p1 = load_exclusions_from_files(EXCLUDED_VALUES_REPO, EXCLUDED_REASONS_REPO)
    v2, p2 = load_exclusions_from_files(EXCLUDED_VALUES_SCENARIO, EXCLUDED_REASONS_SCENARIO)
    return v1 | v2, p1 | p2


def is_excluded(value, reasons, excluded_values, excluded_pairs):
    value_norm = normalize_exclusion_value(value)
    reason_list = [str(r or '').strip().lower() for r in reasons]
    if value_norm in excluded_values:
        return True
    for reason in reason_list:
        if (value_norm, reason) in excluded_pairs:
            return True
    return False


def _parse_env_file(env_path: Path) -> dict:
    data = {}
    if not env_path.exists():
        return data
    with env_path.open('r', encoding='utf-8') as handle:
        for raw in handle:
            line = raw.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            key, value = line.split('=', 1)
            key = key.strip()
            value = value.strip().strip('"').strip("'")
            if key:
                data[key] = value
    return data


def resolve_vt_api_key() -> tuple[str, str]:
    env_path = SCENARIO_DIR / '.env'
    if env_path.exists():
        env_data = _parse_env_file(env_path)
        key = str(env_data.get('VT_API_KEY', '') or '').strip()
        if key:
            return key, 'scenario .env'
    key = str(os.environ.get('VT_API_KEY', '') or '').strip()
    if key:
        return key, 'environment'
    return '', 'missing'


def get_vt_file_verdict(api_key: str, file_hash: str) -> tuple[str, list]:
    h = normalize_vt_file_hash(str(file_hash or ''))
    if not h:
        return '', []
    req = Request(
        f'https://www.virustotal.com/api/v3/files/{h}',
        headers={'x-apikey': api_key},
    )
    try:
        with urlopen(req, timeout=30) as resp:
            payload = json.loads(resp.read().decode('utf-8'))
    except HTTPError as ex:
        if ex.code == 404:
            return 'not_found', []
        if ex.code == 429:
            raise RuntimeError('VirusTotal API rate limit reached (HTTP 429).')
        return 'error', []
    except URLError as ex:
        raise RuntimeError(f'VirusTotal network error: {ex.reason}')

    attrs = payload.get('data', {}).get('attributes', {})
    stats = attrs.get('last_analysis_stats', {}) or {}
    tags = attrs.get('tags', []) or []
    if not isinstance(tags, list):
        tags = []
    malicious = int(stats.get('malicious', 0) or 0)
    suspicious = int(stats.get('suspicious', 0) or 0)
    harmless = int(stats.get('harmless', 0) or 0)
    undetected = int(stats.get('undetected', 0) or 0)
    if malicious > 0:
        return 'malicious', tags
    if suspicious > 0:
        return 'suspicious', tags
    if harmless > 0 and malicious == 0 and suspicious == 0:
        return 'clean', tags
    if undetected > 0:
        return 'undetected', tags
    return 'unknown', tags


PREVALENCE_COL_HINTS = (
    'count', 'prevalence', 'frequency', 'occurrences', 'event_count', 'num_events', 'hits', 'seen',
)


def pick_prevalence_column(columns) -> str | None:
    for c in columns:
        cl = c.lower().strip()
        if cl in PREVALENCE_COL_HINTS:
            return c
        if any(h in cl for h in PREVALENCE_COL_HINTS):
            return c
    return None


def is_aggregate_export(columns, proc_col) -> bool:
    """Power Query aggregate: module.path + count(), no loading-process column."""
    if proc_col is not None:
        return False
    lower = {c.lower() for c in columns}
    return 'module.path' in lower


def normalize_vt_file_hash(value: str) -> str:
    s = str(value or '').strip().lower()
    if len(s) in (32, 40, 64) and re.match(r'^[a-f0-9]+$', s):
        return s
    return ''


def pick_process_path_column(columns) -> str | None:
    priority_exact = (
        'image', 'src.process.image.path', 'event.src.process.image.path',
        'initiatingprocessfilepath', 'initiatingprocessfolderpath',
        'process.image.path',
    )
    lower_map = {c.lower(): c for c in columns}
    for want in priority_exact:
        if want in lower_map:
            return lower_map[want]
    for c in columns:
        cl = c.lower()
        if 'src.process.image.path' in cl or cl.endswith('process.image.path'):
            if 'parent' not in cl:
                return c
    for c in columns:
        cl = c.lower()
        if cl == 'initiatingprocessfilename':
            return c
    return None


def pick_module_path_column(columns) -> str | None:
    priority_exact = (
        'module.path',
        'imageloaded',
        'tgt.file.path',
        'event.tgt.file.path',
        'modulepath',
    )
    lower_map = {c.lower(): c for c in columns}
    for want in priority_exact:
        if want in lower_map:
            return lower_map[want]
    for c in columns:
        cl = c.lower()
        if 'imageloaded' in cl or ('tgt.file' in cl and 'path' in cl):
            return c
    return None


def pick_folder_file_columns(columns):
    # DeviceImageLoadEvents-style FolderPath + FileName.
    lower = {c.lower(): c for c in columns}
    fp = lower.get('folderpath') or lower.get('tgt.file.folderpath')
    fn = lower.get('filename') or lower.get('tgt.file.name') or lower.get('file')
    if fp and fn:
        return fp, fn
    return None, None


def pick_hash_column(columns) -> str | None:
    priority_exact = (
        'sha256',
        'module.sha256',
        'module.sha1',
        'module.md5',
        'sha1',
        'md5',
        'hashes',
        'tgt.file.sha256',
        'filehash',
        'sha-256',
    )
    lower_map = {c.lower(): c for c in columns}
    for want in priority_exact:
        if want in lower_map:
            return lower_map[want]
    for c in columns:
        cl = c.lower()
        if 'module.sha' in cl:
            return c
    return None


def pick_signed_column(columns) -> str | None:
    for c in columns:
        cl = c.lower()
        if cl in ('signed', 'signaturevalid', 'signaturestate', 'tgt.file.issigned'):
            return c
    return None


def pick_hostname_column(columns) -> str | None:
    for c in columns:
        cl = c.lower()
        if cl in ('endpoint.name', 'computer', 'devicename', 'hostname', 'device.id'):
            return c
    return None


def is_target_loader_process(process_path: str) -> bool:
    base = path_basename(process_path)
    if not base:
        return False
    return base.lower() in TARGET_PROCESS_NAMES


def score_path_heuristics(module_path: str) -> tuple[int, list[str]]:
    mpath = normalize_windows_path(module_path)
    score = 0
    reasons: list[str] = []
    if not mpath or not mpath.endswith('.dll'):
        if mpath and not mpath.endswith(('.exe', '.dll', '.ocx', '.sys')):
            score += 1
            reasons.append('non-dll-extension')
    if '\\temp\\' in mpath or '\\appdata\\local\\temp' in mpath:
        score += 3
        reasons.append('temp-path')
    if '\\downloads\\' in mpath:
        score += 2
        reasons.append('downloads-path')
    if mpath.startswith('c:\\users\\') and '\\program files' not in mpath and '\\windows\\' not in mpath:
        if 'teams' not in mpath and 'microsoft' not in mpath and 'edge' not in mpath and 'chrome' not in mpath:
            score += 1
            reasons.append('user-profile-non-vendor-path')
    base = path_basename(mpath)
    if len(base) > 18 and shannon_entropy(base) > 3.4:
        score += 1
        reasons.append('high-entropy-filename')
    return score, reasons


def score_signed(signed_val) -> tuple[int, list[str]]:
    if signed_val is None or (isinstance(signed_val, float) and np.isnan(signed_val)):
        return 0, []
    s = str(signed_val).strip().lower()
    if s in ('false', '0', 'no', 'unsigned', 'invalid'):
        return 2, ['unsigned-or-invalid-signature']
    return 0, []


def score_prevalence_column(val) -> tuple[int, list[str]]:
    if val is None:
        return 0, []
    try:
        if pd.isna(val):
            return 0, []
    except Exception:
        pass
    try:
        v = int(float(val))
    except (TypeError, ValueError):
        return 0, []
    if v <= LOW_PREVALENCE_THRESHOLD:
        return 1, ['low-prevalence-column']
    return 0, []


excluded_values_cfg, excluded_value_reason_cfg = merge_exclusions()
print(f'Exclusion entries: values={len(excluded_values_cfg)}, value+reason={len(excluded_value_reason_cfg)}')


Exclusion entries: values=0, value+reason=0


## EXECUTE — Experimentation Time

- **Gather Data:** Load module-load CSVs from `input/`, map columns to a common schema, detect per-event vs. aggregated input shape.
- **Pre-Process Data:** Normalize paths, filter to target processes, compute fleet-wide rarity counts.
- **Apply:** Score modules using fleet rarity, path heuristics, and optional VT hash reputation; apply exclusions.
- **Analyze:** Review ranked findings above the score threshold.
- **Escalate Critical Findings:** Modules with VT malicious verdicts or very high scores in sensitive paths warrant immediate incident response.

## Load module telemetry from `input/`


In [13]:
csv_paths = sorted(glob.glob(str(INPUT_DIR / '**' / '*.csv'), recursive=True))
print(f'Found {len(csv_paths)} CSV file(s).')

if not csv_paths:
    raise FileNotFoundError(
        f'No CSV files in {_rel(INPUT_DIR)}. Add module-load exports (Image + ImageLoaded, module.path + count(), etc.).'
    )

parts: list[pd.DataFrame] = []
for p in csv_paths:
    sub = pd.read_csv(p, low_memory=False)
    try:
        src_rel = str(Path(p).relative_to(REPO_ROOT)) if Path(p).is_relative_to(REPO_ROOT) else p
    except (ValueError, AttributeError):
        src_rel = p
    sub['_source_file'] = src_rel

    proc_col = pick_process_path_column(sub.columns)
    mod_col = pick_module_path_column(sub.columns)
    fold_col, file_col = pick_folder_file_columns(sub.columns)

    if mod_col:
        sub['_module_path'] = sub[mod_col].astype(str)
    elif fold_col and file_col:
        sub['_module_path'] = (
            sub[fold_col].astype(str).str.rstrip('\\/')
            + '\\'
            + sub[file_col].astype(str)
        )
    else:
        raise ValueError(
            f'{src_rel}: could not detect loaded module path. Columns: {list(sub.columns)}'
        )

    sub['_module_path'] = sub['_module_path'].str.strip()
    sub = sub[sub['_module_path'].str.len() > 0]

    agg = is_aggregate_export(sub.columns, proc_col)
    if agg:
        sub['_agg_mode'] = True
        sub['_loader'] = '(pre-filtered aggregate)'
    else:
        if not proc_col:
            raise ValueError(
                f'{src_rel}: could not detect loading process column. Columns: {list(sub.columns)}'
            )
        sub['_agg_mode'] = False
        sub['_loader'] = sub[proc_col].astype(str)
        sub['_keep'] = sub['_loader'].map(is_target_loader_process)
        sub = sub[sub['_keep']].copy()

    prevalence_col = pick_prevalence_column(sub.columns)
    hash_col = pick_hash_column(sub.columns)
    signed_col = pick_signed_column(sub.columns)
    host_col = pick_hostname_column(sub.columns)

    if prevalence_col is not None:
        sub['_prevalence_val'] = pd.to_numeric(sub[prevalence_col], errors='coerce')
    else:
        sub['_prevalence_val'] = np.nan
    if hash_col and hash_col in sub.columns:
        sub['_hash_val'] = sub[hash_col].astype(str)
    else:
        sub['_hash_val'] = ''
    sub['_signed_val'] = sub[signed_col] if signed_col else None
    sub['_host_val'] = sub[host_col].astype(str) if host_col else ''

    parts.append(sub)

if not parts:
    raise ValueError('No CSV rows after parsing.')

filtered = pd.concat(parts, ignore_index=True)
raw = filtered

print(f'Aggregate mode rows: {int(filtered["_agg_mode"].sum()):,}')
print(f'Per-event mode rows: {int((~filtered["_agg_mode"]).sum()):,}')
print(f'Rows total: {len(filtered):,}')
print(f'Any hash column materialized: {bool((filtered["_hash_val"].astype(str).str.len() > 0).any())}')


Found 2 CSV file(s).
Aggregate mode rows: 44,445
Per-event mode rows: 3
Rows total: 44,448
Any hash column materialized: True


## Score loads (fleet rarity + heuristics) and optional VirusTotal


In [ ]:
fleet_series = filtered.loc[~filtered['_agg_mode'], '_module_path'].astype(str).map(
    normalize_windows_path
)
fleet_counts = fleet_series.value_counts()
hash_vt_cache: dict[str, tuple[str, list]] = {}

vt_key, vt_src = resolve_vt_api_key()
if vt_key:
    unique_hashes = set()
    for h in filtered['_hash_val'].dropna().astype(str):
        hl = normalize_vt_file_hash(str(h))
        if hl:
            unique_hashes.add(hl)
    print(f'VT_API_KEY from {vt_src}; querying {len(unique_hashes)} unique file hash(es)...')
    for i, h in enumerate(sorted(unique_hashes)):
        try:
            verdict, tags = get_vt_file_verdict(vt_key, h)
            hash_vt_cache[h] = (verdict, tags)
        except Exception:
            hash_vt_cache[h] = ('error', [])
        if i % 20 == 19:
            time.sleep(0.2)
else:
    print('VT_API_KEY not set; skipping VirusTotal file lookups.')

findings = []
for idx, row in filtered.iterrows():
    mod_path = str(row['_module_path'])
    mod_norm = normalize_windows_path(mod_path)
    loader = str(row['_loader'])
    agg = bool(row['_agg_mode'])
    reasons: list[str] = []
    score = 0

    s, r = score_path_heuristics(mod_path)
    score += s
    reasons.extend(r)

    if agg:
        if pd.notna(row.get('_prevalence_val')):
            fc = int(float(row['_prevalence_val']))
        else:
            fc = 1
        if fc <= LOW_PREVALENCE_THRESHOLD:
            score += 1
            reasons.append('low-prevalence-count')
    else:
        fc = int(fleet_counts.get(mod_norm, 0))
        if fc <= LOW_PREVALENCE_THRESHOLD:
            score += 1
            reasons.append('fleet-rare-path')
        sp, sr = score_prevalence_column(row.get('_prevalence_val'))
        score += sp
        reasons.extend(sr)

    sp2, sr2 = score_signed(row.get('_signed_val'))
    score += sp2
    reasons.extend(sr2)

    vt_verdict = ''
    vt_tags: list[str] = []
    if vt_key:
        hv = row.get('_hash_val')
        if hv is not None and str(hv).strip():
            hl = normalize_vt_file_hash(str(hv))
            if hl:
                v, t = hash_vt_cache.get(hl, ('', []))
                vt_verdict = v
                vt_tags = t if isinstance(t, list) else []
                if vt_verdict in ('malicious', 'suspicious'):
                    score += 2
                    reasons.append(f'vt-{vt_verdict}')
                elif vt_verdict == 'not_found':
                    score += 1
                    reasons.append('vt-not-found')

    host_val = str(row.get('_host_val', '') or '')
    hash_out = str(row.get('_hash_val', '') or '').strip()

    findings.append({
        'loading_process_path': loader,
        'loading_process_basename': path_basename(loader) if loader and not agg else '(n/a)',
        'aggregate_export': agg,
        'module_path': mod_path,
        'module_path_normalized': mod_norm,
        'fleet_row_count': fc,
        'prevalence_value': row.get('_prevalence_val'),
        'file_hash': hash_out,
        'signed': row.get('_signed_val'),
        'hostname': host_val,
        'score': score,
        'reasons': ';'.join(reasons),
        'vt_verdict': vt_verdict,
        'vt_tags': ';'.join(vt_tags) if vt_tags else '',
        'source_file': row['_source_file'],
    })

findings_df = pd.DataFrame(findings)
if len(findings_df) > 0:
    findings_df = findings_df.sort_values('score', ascending=False).reset_index(drop=True)

# De-duplicate same module_path per source row already unique; apply exclusions and threshold
def row_excluded(rec) -> bool:
    val = rec.get('module_path_normalized') or rec.get('module_path')
    rs = [x.strip() for x in str(rec.get('reasons', '')).split(';') if x.strip()]
    return is_excluded(val, rs, excluded_values_cfg, excluded_value_reason_cfg)

if len(findings_df) > 0:
    mask = findings_df.apply(lambda r: not row_excluded(r.to_dict()), axis=1)
    findings_df = findings_df[mask].reset_index(drop=True)
    findings_df = findings_df[findings_df['score'] >= SCORE_THRESHOLD].reset_index(drop=True)

print(f'Findings with score >= {SCORE_THRESHOLD} (after exclusions): {len(findings_df):,}')
if len(findings_df) > 0:
    display(findings_df.head(20))


VT_API_KEY from environment; querying 11771 unique file hash(es)...


In [ ]:
OUTPUT_COLUMNS = [
    'loading_process_path', 'loading_process_basename', 'aggregate_export', 'module_path',
    'module_path_normalized', 'fleet_row_count', 'prevalence_value', 'file_hash', 'signed',
    'hostname', 'score', 'reasons', 'vt_verdict', 'vt_tags', 'source_file',
]

out_path = OUTPUT_DIR / 'abnormal_dll_loading_results.csv'
top100_path = OUTPUT_DIR / 'abnormal_dll_loading_top100.csv'

if len(findings_df) > 0:
    findings_df.to_csv(out_path, index=False)
    findings_df.head(100).to_csv(top100_path, index=False)
else:
    pd.DataFrame(columns=OUTPUT_COLUMNS).to_csv(out_path, index=False)
    pd.DataFrame(columns=OUTPUT_COLUMNS).to_csv(top100_path, index=False)

print(f'Wrote { _rel(out_path) }')
print(f'Wrote { _rel(top100_path) }')

meta = {
    'scenario': 'abnormal_dll_loading_common_applications',
    'ref': 'M25',
    'rows_input': int(len(raw)),
    'rows_aggregate_export': int(filtered['_agg_mode'].sum()),
    'rows_per_event_export': int((~filtered['_agg_mode']).sum()),
    'findings_count': int(len(findings_df)),
    'score_threshold': SCORE_THRESHOLD,
    'vt_key_present': bool(vt_key),
}
meta_path = OUTPUT_DIR / 'run_metadata.json'
meta_path.write_text(json.dumps(meta, indent=2), encoding='utf-8')
print(f'Wrote { _rel(meta_path) }')


## ACT — Wrapping Up the Investigation

- **Document Findings:** KPI chart and ranked results summarize hunt outcomes — module counts, score distributions, VT verdicts.
- **Preserve Hunt:** Results CSV, top-100 export, run metadata JSON, and KPI chart PNG archived to `output/`.
- **Create Detections / Playbooks:** High-confidence side-loading findings can feed endpoint detection rules or application control policies.

## KPI chart


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle('M25 — Abnormal DLL loading (common applications)', fontsize=14)

if len(findings_df) > 0:
    ax = axes[0, 0]
    findings_df['loading_process_basename'].value_counts().head(10).plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Top loading processes (basename)')
    ax = axes[0, 1]
    sc = findings_df['score'].value_counts().sort_index()
    sc.plot(kind='bar', ax=ax, color='coral')
    ax.set_title('Score distribution')
    ax = axes[1, 0]
    reasons_exp = []
    for r in findings_df['reasons'].fillna(''):
        reasons_exp.extend([x.strip() for x in str(r).split(';') if x.strip()])
    if reasons_exp:
        pd.Series(reasons_exp).value_counts().head(12).plot(kind='barh', ax=ax, color='seagreen')
    ax.set_title('Top reasons')
    ax = axes[1, 1]
    vc = findings_df['vt_verdict'].replace('', 'n/a').value_counts()
    vc.plot(kind='pie', ax=ax, autopct='%1.0f%%')
    ax.set_title('VT verdict (if present)')
else:
    for ax in axes.flat:
        ax.text(0.5, 0.5, 'No findings', ha='center', va='center')
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
chart_path = OUTPUT_DIR / 'abnormal_dll_loading_kpis.png'
plt.savefig(chart_path, dpi=120, bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(chart_path)))
print(f'Saved chart to {_rel(chart_path)}')


## KNOWLEDGE — Continuous Improvement

- **Re-Add Topics to Backlog:** New side-loading techniques, target processes, or suspicious path patterns discovered during analysis become future hunt hypotheses.
- **Communicate Findings:** Share confirmed DLL abuse with endpoint security, application owners, and SOC leadership.
- **Feed Improvements Back:** Add confirmed benign modules to exclusion lists; expand `TARGET_PROCESS_NAMES` as new high-value applications are identified; update path heuristics for emerging attack patterns.
- **Measure Effectiveness:** Compare finding counts, VT hit rates, and exclusion growth across successive runs to track detection maturity.